# Lab 12: Faster R-CNN Inference

**Module 12** | Read `notes/12-faster-rcnn.pdf` before running this notebook.

Run a pre-trained Faster R-CNN on sample images, filter detections by score, and save annotated outputs.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Faster R-CNN object detection

**Object detection** finds *where* objects are in an image and *what* they are. Unlike classification (one label per image), detection outputs **bounding boxes** (rectangles around objects) plus a class name and confidence score for each box.

**Faster R-CNN** is a classic two-stage detector:
1. A backbone (ResNet-50 + FPN) extracts image features.
2. A Region Proposal Network suggests candidate box locations.
3. A classification head refines each region and assigns a COCO class label.

This lab loads COCO pre-trained weights, runs inference on local sample images, filters detections with confidence above 0.5, prints per-image details, saves annotated copies, and previews one result inline.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **Object detection** | A vision task that locates objects with bounding boxes and assigns a class label to each box. |
| **Bounding box** | A rectangle `(x1, y1, x2, y2)` enclosing an object, where `(x1, y1)` is the top-left corner and `(x2, y2)` is the bottom-right. |
| **Confidence score** | A number between 0 and 1 indicating how sure the model is about a detection. Higher means more confident. |
| **IoU** | Intersection over Union. Measures overlap between two boxes: area of overlap divided by area of union. IoU = 1 means perfect overlap; IoU = 0 means no overlap. |
| **mAP** | mean Average Precision. A standard detection metric that averages precision across classes and IoU thresholds. Higher mAP means better detection quality. |
| **COCO** | Common Objects in Context. A large detection dataset with 80 object categories (person, car, dog, etc.). |

Refer back to this table whenever a term appears in code or output.


### Step 1: Load the pre-trained model

**What this section does:** Downloads (if needed) Faster R-CNN with ResNet-50 FPN backbone, moves it to `device`, and locates sample JPG images.

**Why we do it:** Pre-trained weights let you run detection immediately without training. The weight bundle includes preprocessing transforms and the COCO category name list used to decode label indices.


**What to look for in the output**

- `Model: Faster R-CNN ResNet-50 FPN (COCO pre-trained)`.
- `COCO categories: 91` (includes background index 0).
- At least one sample JPG found in the images directory.
- First 10 category names look like real COCO labels (person, bicycle, car, ...).


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)
from runtime_env import ensure_sample_images

ROOT = Path("..").resolve()
IMAGE_DIR = ensure_sample_images()
OUT_DIR = ROOT / "labs" / "annotated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# DEFAULT weights include both network parameters and the expected image transforms.
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval().to(device)
preprocess = weights.transforms()
category_names = weights.meta["categories"]

image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
print(f"Model: Faster R-CNN ResNet-50 FPN (COCO pre-trained)")
print(f"COCO categories: {len(category_names)}")
print(f"Sample images found: {len(image_paths)} in {IMAGE_DIR}")
if not image_paths:
    raise FileNotFoundError(
        f"No JPG files in {IMAGE_DIR}. Check your network connection or run: python download_datasets.py"
    )
print()
print("First 10 category names:", category_names[:10])


### Step 2: Run inference and record detections

**What this section does:** For each image, preprocesses pixels, runs the model, keeps detections with `score > 0.5`, prints class name, confidence, and box coordinates, then draws green rectangles and saves annotated PNGs.

**Why we do it:** The raw model output is a large tensor of candidate boxes. Filtering by score removes low-confidence guesses. Printing coordinates lets you verify boxes without opening image files first.


**What to look for in the output**

- Per-image header with filename and dimensions.
- Each kept detection shows class name, score (three decimals), and four box coordinates.
- `(no detections above threshold)` is OK for plain or unusual images.
- `Saved N annotated images` at the end with N equal to the number of input JPGs.


In [ ]:
SCORE_THRESHOLD = 0.5
saved: list = []
summary_rows: list[dict] = []

try:
    font = ImageFont.truetype("arial.ttf", 16)
except OSError:
    font = ImageFont.load_default()

for img_path in image_paths:
    image = Image.open(img_path).convert("RGB")
    tensor = preprocess(image).to(device)
    with torch.no_grad():
        prediction = model([tensor])[0]

    draw = ImageDraw.Draw(image)
    detections: list[dict] = []

    print(f"\n{img_path.name} ({image.size[0]}x{image.size[1]})")
    print("-" * 70)

    # Zip boxes, class indices, and confidence scores together.
    for box, label, score in zip(
        prediction["boxes"], prediction["labels"], prediction["scores"]
    ):
        score_val = score.item()
        if score_val <= SCORE_THRESHOLD:
            continue
        x1, y1, x2, y2 = box.tolist()
        name = category_names[label.item()]
        detections.append({
            "class": name,
            "score": score_val,
            "box": (x1, y1, x2, y2),
        })
        print(
            f"  {name:20s}  score={score_val:.3f}  "
            f"box=[{x1:6.1f}, {y1:6.1f}, {x2:6.1f}, {y2:6.1f}]"
        )
        draw.rectangle([x1, y1, x2, y2], outline="lime", width=3)
        draw.text((x1, max(y1 - 18, 0)), f"{name} {score_val:.2f}", fill="yellow", font=font)

    if not detections:
        print("  (no detections above threshold)")

    out_path = OUT_DIR / f"{img_path.stem}_det.jpg"
    image.save(out_path)
    saved.append(out_path)

    classes_found = sorted({d["class"] for d in detections})
    summary_rows.append({
        "image": img_path.name,
        "detections": len(detections),
        "classes": ", ".join(classes_found) if classes_found else "(none)",
        "output": out_path.name,
    })

print(f"\nSaved {len(saved)} annotated images to {OUT_DIR}")


### Step 3: Summary table across all images

**What this section does:** Prints a compact table: image name, detection count, class list, and output filename for every processed image.

**Why we do it:** A table helps you spot images with many detections or unexpected class labels before opening annotated files one by one.


**What to look for in the output**

- Header row with Image, Count, Classes, Output columns.
- Total detections summed across all images.
- Class names in the table should match what was printed per image in Step 2.


In [ ]:
print("Detection summary (score > {:.1f}):".format(SCORE_THRESHOLD))
print("=" * 90)
print(f"{'Image':<24} {'Count':>6}  {'Classes':<50} {'Output':<12}")
print("-" * 90)
for row in summary_rows:
    classes = row["classes"]
    if len(classes) > 48:
        classes = classes[:45] + "..."
    print(
        f"{row['image']:<24} {row['detections']:>6}  "
        f"{classes:<50} {row['output']:<12}"
    )
print("-" * 90)
total_dets = sum(r["detections"] for r in summary_rows)
print(f"Total detections kept: {total_dets} across {len(summary_rows)} images")


### Step 4: Preview one result inline

**What this section does:** Displays the first annotated image inside the notebook with matplotlib.

**Why we do it:** Numbers and coordinates are useful, but you should also verify visually that boxes align with real objects and labels are readable.


**What to look for in the output**

- A figure appears with the annotated image.
- Green boxes should surround visible objects.
- Yellow text labels should sit near the top of each box.


In [ ]:
import matplotlib.pyplot as plt

preview = Image.open(saved[0])
plt.figure(figsize=(10, 7))
plt.imshow(preview)
plt.axis("off")
plt.title(f"{saved[0].name} (inline preview)")
plt.show()


### Step 5: Evaluation checklist

**What this section does:** Prints counts of categories, images, and unique detected classes.

**Why we do it:** Confirms the pipeline ran end-to-end. False positives often appear on cluttered backgrounds; raising `SCORE_THRESHOLD` reduces them at the cost of missed objects.


**What to look for in the output**

- COCO categories loaded: 91.
- Images processed matches the number of input JPGs.
- Unique classes detected is a small subset of 80 COCO object classes.
- Annotated output path printed.


In [ ]:
all_classes = set()
for row in summary_rows:
    if row["classes"] != "(none)":
        all_classes.update(row["classes"].split(", "))

print("Evaluation checklist:")
print(f"  COCO categories loaded: {len(category_names)}")
print(f"  Images processed: {len(summary_rows)}")
print(f"  Unique classes detected: {len(all_classes)}")
if all_classes:
    print(f"  Classes seen: {sorted(all_classes)}")
print(f"  Annotated outputs in: {OUT_DIR}")
